In [1]:
import sys
sys.path.insert(0, '../lib')

import glob
import os
import pickle
import datetime

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import datasets
import numpy as np
import pandas as pd
import wandb
import scipy
import sklearn.metrics
import torch
import torch.distributions as td

import common_data

/projects/b1196/envs/serniczek/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%config InlineBackend.figure_format = "retina"

In [3]:
pd.options.display.max_columns = 300
pd.options.display.max_rows = 350
pd.options.display.max_colwidth = 10000

In [4]:
mpl.rc_file_defaults()

In [5]:
# Isolate the sample ids of the rare pathogen samples
sc_labels = pd.read_csv(common_data.SC_LABELS)
filtered_groups = [
    'Gram-*; SARS-CoV-2',
    'Gram-*; SARS-CoV-2; Gram+',
    'Other viruses',
    'Gram-*; Pseudomonas aeruginosa',
    'Pseudomonas aeruginosa; Gram+',
    'Other viruses; Pseudomonas aeruginosa',
    'Gram-*; Other viruses; SARS-CoV-2'
]
rare_pathogen = list(sc_labels.bal_barcode[sc_labels.pathogen_groups.isin(filtered_groups)])

In [6]:
TOKENS_DIR = common_data.GENEFORMER_DATA / 'tokens'
DATA_DIR = common_data.GENEFORMER_DATA / 'predictions'
N_SPLITS = 4


def load_data(token_path):
    # Filter files with prefix "chunk"
    chunks = glob.glob(f'{token_path}/chunk*.dataset')
    tokenized_chunks = [datasets.load_from_disk(file) for file in chunks]
    return datasets.concatenate_datasets(tokenized_chunks)


def filter_split_data(data):
    idx = []
    for individual in data['individual']:
        idx.append(individual in rare_pathogen)
    test = data.select(np.where(np.array(idx))[0])
    test = test.add_column('label', [1] * len(test))
    return test

In [7]:
dataset = load_data(TOKENS_DIR)

In [8]:
def get_classification_entropy(classification_probs, norm=True):
    dist = td.Categorical(probs=classification_probs)
    classification_entropy = dist.entropy()
    if norm:
        classification_entropy = classification_entropy / torch.log(input=torch.Tensor([2.]))
    return classification_entropy

### Show data types and structure before processing everything

Files loaded here are generated by this script: `scripts/run_predict_rare_pathogen.py` and this script: `scripts/save_avg_mc_probs.py` via the Snakefile

Function `filter_split_data` loads rare pathogen samples

In [ ]:
npc_data = filter_split_data(dataset)
orig_cell_ids = None

rare_pathogen_preds = {}
for task_path in common_data.get_tasks(context='geneformer'):
    if not task_path.startswith('H_') and not task_path.startswith('viral'):
        continue
    for split_path in [f'split_{i + 1}' for i in range(N_SPLITS)]:
        if not (DATA_DIR / task_path / split_path).exists():
            continue
        task_info = common_data.get_task_info(task_path, split_path)
        for model_path in (DATA_DIR / task_path / split_path).iterdir():
            if not model_path.name.startswith('model_'):
                continue
            if not (model_path / 'rare_pathogen_predictions/avg_probs.npy').exists():
                continue

            print(f'Loading data for model {model_path}')
            cell_ids = np.load(model_path / 'rare_pathogen_predictions/cell_ids.npy')
            if orig_cell_ids is None:
                orig_cell_ids = cell_ids

                print(f'Cell ids (custom order of test cells) shape {cell_ids.shape}, first value: {cell_ids[0]}')
                # Create a dictionary mapping cell_ids to their indices
                cell_id_to_index = {cell_id: index for index, cell_id in enumerate(cell_ids)}
                npc_data = npc_data.map(lambda x: {"sort_index": cell_id_to_index[x['obs_names']]})
                npc_data = npc_data.sort("sort_index")
                npc_data = npc_data.remove_columns("sort_index")
                print(f'NPC data shape {len(npc_data)}, first cell_id {npc_data["obs_names"][0]}')

            if not np.all(cell_ids == orig_cell_ids):
                print(f'!!! Cell ids mismatch for {model_path}')
                raise

            probs = np.load(model_path / 'rare_pathogen_predictions/avg_probs.npy')
            print(f'Loaded probabilities shape {probs.shape}, all sum to 1: {np.allclose(probs.sum(axis=1), 1)}')
            preds = np.argmax(probs, axis=-1)
            entropy = get_classification_entropy(torch.tensor(probs))

            model_df = pd.DataFrame(dict(
                true_label=npc_data['label'],
                cell_types=npc_data['Level_6'],
                obs_names=npc_data['obs_names'],
                sample=npc_data['individual'],
                pred_label=preds,
                entropy=entropy,
                pos_class_prob=probs[:, 1]
            ))
            rare_pathogen_preds[str(model_path)] = model_df
            break
        break
    break

Loading data for model /gpfs/projects/b1196/ewa_group/serniczek/08_geneformer/../data/08_geneformer/predictions/viral_vs_bacterial/split_1/model_2
Cell ids (custom order of test cells) shape (154590,), first value: SC594_GAGTCCGCATTAGGCT
NPC data shape 154590, first cell_id SC594_GAGTCCGCATTAGGCT
Loaded probabilities shape (154590, 2), all sum to 1: True


In [10]:
%%time

rare_pathogen_preds = {}
for task_path in common_data.get_tasks(context='geneformer'):
    if not task_path.startswith('H_') and not task_path.startswith('viral'):
        continue
    for split_path in [f'split_{i + 1}' for i in range(N_SPLITS)]:
        if not (DATA_DIR / task_path / split_path).exists():
            continue
        task_info = common_data.get_task_info(task_path, split_path)
        for model_path in (DATA_DIR / task_path / split_path).iterdir():
            if not model_path.name.startswith('model_'):
                continue
            if not (model_path / 'rare_pathogen_predictions/avg_probs.npy').exists():
                continue

            # print(f'Loading data for model {model_path}')
            cell_ids = np.load(model_path / 'rare_pathogen_predictions/cell_ids.npy')
            if not np.all(cell_ids == orig_cell_ids):
                print(f'!!! Cell ids mismatch for {model_path}')
                raise

            probs = np.load(model_path / 'rare_pathogen_predictions/avg_probs.npy')
            # print(f'Loaded probabilities shape {probs.shape}, all sum to 1: {np.allclose(probs.sum(axis=1), 1)}')
            preds = np.argmax(probs, axis=-1)
            entropy = get_classification_entropy(torch.tensor(probs))

            model_df = pd.DataFrame(dict(
                true_label=npc_data['label'],
                cell_types=npc_data['Level_6'],
                obs_names=npc_data['obs_names'],
                sample=npc_data['individual'],
                pred_label=preds,
                entropy=entropy,
                pos_class_prob=probs[:, 1]
            ))
            rare_pathogen_preds[str(model_path)] = model_df
            print('.', flush=True, end='')
    print('', flush=True)
    print(task_path + ' done')

....................
viral_vs_bacterial done
....................
H_vs_eCOVID done
....................
H_vs_lCOVID done
....................
H_vs_G+ done
....................
H_vs_G- done
....................
H_vs_P done
....................
H_vs_eCOVID_G+ done
....................
H_vs_lCOVID_G+ done
....................
H_vs_P_COVID done
....................
H_vs_G-_G+ done
CPU times: user 5min 40s, sys: 6.82 s, total: 5min 47s
Wall time: 5min 45s


In [11]:
with open('04_rare_pathogen_preds.pkl', 'wb') as f:
    pickle.dump(rare_pathogen_preds, f)